In [0]:
# Import and Extraction Data (E)
from pyspark.sql.functions import *
from pyspark.sql.types import *

df_customers = spark.read.table("databricks_simulated_retail_customer_data.v01.customers")
df_sales = spark.read.table("databricks_simulated_retail_customer_data.v01.sales")
df_sales_orders = spark.read.table("databricks_simulated_retail_customer_data.v01.sales_orders")

In [0]:
# Transform Step (T) -> Remove duplicates and remove null values

df_sales.count() # 360
df_sales_orders.count() # 4074
df_customers.count() # 28813

# Any of above has ID null on those columns validations, so it didn't have any null 1row to clean

for column in df_customers.columns:
    print(column, df_customers.filter(col(column).isNull()).count())

# It's the unique that can drop duplicates, once the other two tables didn't have their own ID column, only foreign keys

# df_customers = df_customers.dropDuplicates(["customer_id"]).count() -> 28670

df_customers = df_customers.dropDuplicates(["customer_id"])


customer_id 0
tax_id 19389
tax_code 19389
customer_name 0
state 0
city 4765
postcode 0
street 0
number 123
unit 25613
region 15152
district 16732
lon 0
lat 0
ship_to_address 0
valid_from 0
valid_to 27380
units_purchased 0
loyalty_segment 0


In [0]:
# Transform Step (T) -> normalize null values on coluns, and change data types (if needed)

#df_customers
#  |-- customer_id: long
#  |-- tax_id: long
#  |-- tax_code: string w size 2
#  |-- customer_name: string
#  |-- state: string
#  |-- city: string
#  |-- postcode: string
#  |-- street: string
#  |-- number: string
#  |-- unit: string
#  |-- region: string
#  |-- district: string
#  |-- lon: double
#  |-- lat: double
#  |-- ship_to_address: string
#  |-- valid_from: long
#  |-- valid_to: long
#  |-- units_purchased: long
#  |-- loyalty_segment: long

df_customers = df_customers.withColumn("tax_id", col("tax_id").cast("long")).withColumn("tax_code", lpad(col("tax_code"), 2, " ")).withColumn("valid_to", col("valid_to").cast("long"))
df_customers.printSchema()


root
 |-- customer_id: long (nullable = true)
 |-- tax_id: long (nullable = true)
 |-- tax_code: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- street: string (nullable = true)
 |-- number: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- region: string (nullable = true)
 |-- district: string (nullable = true)
 |-- lon: double (nullable = true)
 |-- lat: double (nullable = true)
 |-- ship_to_address: string (nullable = true)
 |-- valid_from: long (nullable = true)
 |-- valid_to: long (nullable = true)
 |-- units_purchased: long (nullable = true)
 |-- loyalty_segment: long (nullable = true)



In [0]:
# Transform Step (T) -> normalize null values on coluns, and change data types (if needed)
#df_sales
#  |-- customer_id: long (nullable = true)
#  |-- customer_name: string (nullable = true)
#  |-- product_name: string (nullable = true)
#  |-- product_category: string (nullable = true)
#  |-- product: string (nullable = true)
#  |-- total_price: long (nullable = true)
#  |-- Month: string (nullable = true)
#  |-- Day: string (nullable = true)
#  |-- Year: string (nullable = true)

df_sales = df_sales.withColumn("Month", substring(col("order_date"), -5, 2)).withColumn("Day", substring(col("order_date"), -2, 2)).withColumn("Year", substring(col("order_date"), 1, 4)).drop("order_date")
df_sales.printSchema()


root
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- total_price: long (nullable = true)
 |-- Month: string (nullable = true)
 |-- Day: string (nullable = true)
 |-- Year: string (nullable = true)



In [0]:
# Transform Step (T) -> normalize null values on coluns, and change data types (if needed)
#df_sales_orders
#  |-- clicked_items: string (nullable = true)
#  |-- customer_id: long (nullable = true)
#  |-- customer_name: string (nullable = true)
#  |-- number_of_line_items: long (nullable = true)
#  |-- order_number: long (nullable = true)
#  |-- ordered_products: string (nullable = true)
#  |-- promo_info: string (nullable = true)
#  |-- Month: string (nullable = true)
#  |-- Day: string (nullable = true)
#  |-- Year: string (nullable = true)
#  |-- Hour: string (nullable = true)
#  |-- Minute: string (nullable = true)

df_sales_orders = df_sales_orders.withColumn("order_datetime",from_unixtime(col("order_datetime"), "yyyy-MM-dd HH:mm")).withColumn("Month", substring(col("order_datetime"), 6, 2)).withColumn("Day", substring(col("order_datetime"), 9, 2)).withColumn("Year", substring(col("order_datetime"), 1, 4)).withColumn("Hour", substring(col("order_datetime"), -5, 2)).withColumn("Minute", substring(col("order_datetime"), -2, 2)).drop("order_datetime")
df_sales_orders.printSchema()


root
 |-- clicked_items: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- number_of_line_items: long (nullable = true)
 |-- order_number: long (nullable = true)
 |-- ordered_products: string (nullable = true)
 |-- promo_info: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- Day: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Hour: string (nullable = true)
 |-- Minute: string (nullable = true)

